# Session 10: Advanced PINN topics (Part 2)

[Session 9](Session9.ipynb) examined how to tune the physics loss weight λ systematically. This session continues the advanced-topics thread by addressing three fundamental limitations that practitioners encounter when scaling physics-informed neural networks (PINNs) beyond simple textbook problems:

1. **Spectral bias** — standard multilayer perceptrons (MLPs) learn low-frequency functions much faster than high-frequency ones, which causes failures when the target solution varies rapidly.
2. **Adaptive collocation sampling** — placing collocation points uniformly wastes computational budget in regions where the partial differential equation (PDE) residual is already small; residual-based refinement focuses effort where it is most needed.
3. **Domain decomposition (XPINN sketch)** — long time domains and complex geometries benefit from splitting the problem into overlapping or adjacent sub-domains that are solved in parallel and coupled at interfaces.

The session closes with a conceptual overview of three recent research directions: conservative PINNs (cPINNs), multi-fidelity PINNs, and Physics-Informed Kolmogorov–Arnold Networks (PIKANs).

**Prerequisites**: [Session 9](Session9.ipynb), PyTorch autograd, and basic Fourier analysis at the level of a second-year physics course.

## 1. Spectral bias and high-frequency failures

### 1.1 The frequency principle

A striking empirical observation, formalised theoretically by Rahaman et al. (2019) and Xu et al. (2020) under the name the *Frequency Principle* (or *spectral bias*), is that gradient-descent training of standard neural networks proceeds in a low-to-high frequency order: components of the target function that oscillate slowly are captured first, whilst rapidly oscillating components are learned orders of magnitude more slowly, if at all.

To see why this matters for PINNs, consider a one-dimensional boundary value problem whose solution contains high-frequency oscillations — for instance a rapidly varying forcing term, a fine-scale material property, or a solution near a wave front. Standard MLP architectures will fail to represent these features accurately within a reasonable training budget.

### 1.2 Fourier perspective

A network $u_\theta(x)$ trained by gradient descent can be analysed in the spectral domain. The *neural tangent kernel* (NTK) framework shows that the convergence rate for frequency $k$ scales as the corresponding eigenvalue $\lambda_k$ of the NTK. For standard MLPs with smooth activations (e.g. tanh), $\lambda_k$ decreases rapidly with $k$, so high-$k$ modes converge slowly.

Concretely, if the target function is $u^*(x) = \sin(\omega x)$, the number of gradient steps required to achieve a given accuracy scales roughly as $\omega^2$ for a shallow network. At $\omega = 10$ this is already 100 times slower than at $\omega = 1$.

### 1.3 Mitigation: Fourier feature embeddings

Tancik et al. (2020) proposed mapping the input coordinates through a bank of sinusoidal features before passing them to the MLP:

$$\gamma(x) = \left[\cos(2\pi B_1 x),\; \sin(2\pi B_1 x),\; \ldots,\; \cos(2\pi B_m x),\; \sin(2\pi B_m x)\right]$$

where $B_1, \ldots, B_m$ are frequencies drawn from a distribution (commonly Gaussian with standard deviation $\sigma$ that sets the frequency scale). This is called a *random Fourier feature* (RFF) embedding. By pre-encoding the desired frequency content into the input, the MLP no longer needs to learn those oscillations itself — they are already present as explicit features.

**Key hyperparameter**: $\sigma$ (the standard deviation of the frequency distribution). Set $\sigma$ to be of the same order as the dominant frequency of the target solution. Too small and the embedding provides no benefit; too large and the network overfits noise.

### 1.4 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(0)
torch.manual_seed(0)

# ── Target function ───────────────────────────────────────────────────────────
OMEGA   = 10.0          # angular frequency of the target sin(ω x)
N_TRAIN = 200           # collocation / training points
EPOCHS  = 5000
LR      = 5e-4

x_np   = np.linspace(0, 1, N_TRAIN).reshape(-1, 1).astype(np.float32)
y_np   = np.sin(OMEGA * x_np)
x_t    = torch.from_numpy(x_np)
y_t    = torch.from_numpy(y_np)

x_eval = np.linspace(0, 1, 500).reshape(-1, 1).astype(np.float32)
y_eval = np.sin(OMEGA * x_eval)
x_eval_t = torch.from_numpy(x_eval)

print(f"Target: sin({OMEGA:.0f}x),  x ∈ [0, 1]")
print(f"Training points: {N_TRAIN},  epochs: {EPOCHS}")

### 1.5 Standard MLP vs Fourier-feature MLP

In [ ]:
# ── Standard MLP (no embedding) ───────────────────────────────────────────────
class StandardMLP(nn.Module):
    """Four-layer MLP: scalar x → scalar u.  Width 64, tanh activations."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1,  64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.net(x)


# ── Fourier-feature MLP ───────────────────────────────────────────────────────
class FourierFeatureMLP(nn.Module):
    """
    Random Fourier feature embedding followed by an MLP.

    The embedding maps scalar x to a 2m-dimensional vector:
        [cos(2π B x), sin(2π B x)]   where B ~ N(0, sigma^2).
    The MLP then acts on this enriched representation.
    """
    def __init__(self, m=64, sigma=8.0):
        super().__init__()
        # Fixed random frequencies — not trained
        B = torch.randn(1, m) * sigma
        self.register_buffer('B', B)
        self.net = nn.Sequential(
            nn.Linear(2 * m, 64), nn.Tanh(),
            nn.Linear(64,    64), nn.Tanh(),
            nn.Linear(64,    64), nn.Tanh(),
            nn.Linear(64,     1),
        )

    def embed(self, x):
        """Map x (N,1) → (N, 2m) via Fourier features."""
        proj = 2 * np.pi * (x @ self.B)       # (N, m)
        return torch.cat([torch.cos(proj), torch.sin(proj)], dim=1)

    def forward(self, x):
        return self.net(self.embed(x))


def train_model(model, x_in, y_in, epochs, lr):
    """Train model with Adam; return list of MSE losses per epoch."""
    opt  = optim.Adam(model.parameters(), lr=lr)
    mse  = nn.MSELoss()
    hist = []
    for _ in range(epochs):
        opt.zero_grad()
        loss = mse(model(x_in), y_in)
        loss.backward()
        opt.step()
        hist.append(loss.item())
    return hist


print("Training standard MLP…")
mlp_std  = StandardMLP()
hist_std = train_model(mlp_std, x_t, y_t, EPOCHS, LR)

print("Training Fourier-feature MLP…")
mlp_ff   = FourierFeatureMLP(m=64, sigma=8.0)
hist_ff  = train_model(mlp_ff,  x_t, y_t, EPOCHS, LR)

with torch.no_grad():
    pred_std = mlp_std(x_eval_t).numpy()
    pred_ff  = mlp_ff(x_eval_t).numpy()

mse_std = float(np.mean((pred_std - y_eval)**2))
mse_ff  = float(np.mean((pred_ff  - y_eval)**2))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: predictions
axes[0].plot(x_eval, y_eval,   'k-',  lw=1.5, label='Exact')
axes[0].plot(x_eval, pred_std, 'r--', lw=1.5, label=f'Standard MLP  (MSE={mse_std:.4f})')
axes[0].plot(x_eval, pred_ff,  'b-',  lw=1.5, label=f'Fourier-feature MLP  (MSE={mse_ff:.4f})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('u(x)')
axes[0].set_title(f'Approximating $\\sin({OMEGA:.0f}x)$', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Centre: training loss
axes[1].semilogy(hist_std, 'r', lw=1.5, label='Standard MLP')
axes[1].semilogy(hist_ff,  'b', lw=1.5, label='Fourier-feature MLP')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MSE loss (log scale)')
axes[1].set_title('Training loss', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Right: pointwise error
axes[2].plot(x_eval, np.abs(pred_std - y_eval), 'r', lw=1.2, label='Standard MLP')
axes[2].plot(x_eval, np.abs(pred_ff  - y_eval), 'b', lw=1.2, label='Fourier-feature MLP')
axes[2].set_xlabel('x')
axes[2].set_ylabel('|prediction − exact|')
axes[2].set_title('Pointwise absolute error', fontsize=12, fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(alpha=0.3)

plt.suptitle(f'Spectral bias demonstration  —  target sin({OMEGA:.0f}x)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal MSE — standard MLP     : {mse_std:.6f}")
print(f"Final MSE — Fourier-feature  : {mse_ff:.6f}")
print(f"Speed-up factor (MSE ratio)  : {mse_std / max(mse_ff, 1e-10):.1f}×")

### 1.6 What the plots show

The standard MLP struggles to follow the ten full cycles of $\sin(10x)$ in $[0,1]$ — it typically converges to a smooth, low-frequency approximation that misses most of the oscillation. The Fourier-feature MLP, by contrast, captures the correct shape because the sinusoidal basis is already encoded in the input layer.

**Choosing $\sigma$**: the distribution of frequencies $B_j \sim \mathcal{N}(0, \sigma^2)$ should cover the dominant frequency of the target. For $\sin(10x)$ the natural frequency is $10/(2\pi) \approx 1.6$ cycles per unit length. Setting $\sigma \approx 8$ puts the median sampled frequency comfortably in this range. In practice, try $\sigma \in \{1, 3, 8, 15\}$ and choose based on validation loss.

**In a PINN context**: replace the first linear layer of any MLP architecture with a Fourier-feature embedding. The rest of the training loop is identical — autograd differentiation through `cos` and `sin` is straightforward. This is particularly valuable for problems such as wave propagation or acoustic scattering where the physical solution inherently contains high-frequency content.

## 2. Adaptive collocation sampling

### 2.1 The problem with uniform sampling

When collocation points are drawn uniformly from the domain, computational effort is distributed evenly regardless of where the PDE residual is large. In problems with sharp gradients or localised features — shock waves, boundary layers, steep fronts — the network devotes an equal number of gradient evaluations to regions where it is already accurate and to regions where it is not. This is wasteful.

### 2.2 Residual-based refinement

The idea of *adaptive collocation* (Lu et al., 2021 — RAR; Daw et al., 2023 — RRALS) is simple:

1. Train for $N_{\text{warm}}$ epochs with a fixed set of collocation points.
2. Evaluate $|r(x_i)|$ — the absolute PDE residual — at a large candidate pool of points.
3. Add the $k$ points with the largest residuals to the collocation set.
4. Continue training with the augmented set.
5. Repeat until convergence or a budget limit is reached.

Formally, if $r(x; \theta) = \mathcal{N}[u_\theta](x) - f(x)$ is the PDE residual, the new points are selected as

$$\mathcal{X}^{\text{new}} = \underset{x \in \mathcal{C}}{\operatorname{top-}k}\; |r(x; \theta)|$$

where $\mathcal{C}$ is a dense candidate pool. The collocation set grows over time, which increases cost per epoch but greatly reduces the total number of epochs required.

### 2.3 Test problem

We use a simple 1D ODE with a steep feature:

$$u'(x) + \alpha \, u(x) = 0, \quad x \in [0, 1], \quad u(0) = 1$$

with exact solution $u(x) = e^{-\alpha x}$. For $\alpha = 20$ the solution decays by a factor of $e^{20} \approx 5 \times 10^8$ across the domain, with most variation concentrated near $x = 0$. Uniform sampling places most points in the flat tail region where the residual is trivially small.

In [ ]:
# ── Adaptive collocation sampling demo ────────────────────────────────────────
torch.manual_seed(1)
np.random.seed(1)

ALPHA      = 20.0       # decay rate — steep near x=0
N_INIT     = 20         # initial collocation points
N_CAND     = 500        # candidate pool for residual evaluation
N_ADD      = 10         # points to add per refinement step
N_ROUNDS   = 5          # number of refinement rounds
EPOCHS_PER = 2000       # epochs between refinements
LR_ADAPT   = 1e-3

# ── Exact solution and evaluation grid ────────────────────────────────────────
x_ev  = np.linspace(0, 1, 400).astype(np.float32)
u_ev  = np.exp(-ALPHA * x_ev)

# ── Network (same architecture for both runs) ─────────────────────────────────
def make_net():
    return nn.Sequential(
        nn.Linear(1, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 32), nn.Tanh(),
        nn.Linear(32, 1),
    )


def ode_residual(net, x_col):
    """Residual of u' + alpha*u = 0 at collocation points x_col (tensor, requires_grad)."""
    x = x_col.clone().requires_grad_(True)
    u = net(x)
    du = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u),
                             create_graph=True)[0]
    return du + ALPHA * u


def train_step(net, opt, x_col_t, x_ic_t, u_ic_t):
    """One epoch of training: physics loss + IC loss."""
    opt.zero_grad()
    res   = ode_residual(net, x_col_t)
    l_pde = torch.mean(res**2)
    l_ic  = torch.mean((net(x_ic_t) - u_ic_t)**2)
    loss  = l_pde + 10.0 * l_ic
    loss.backward()
    opt.step()
    return loss.item()


# ── Initial condition ─────────────────────────────────────────────────────────
x_ic_t = torch.tensor([[0.0]], dtype=torch.float32)
u_ic_t = torch.tensor([[1.0]], dtype=torch.float32)

# ══════════════════════════════════════════════════════════════════════════════
#  Run 1: uniform collocation
# ══════════════════════════════════════════════════════════════════════════════
net_uni  = make_net()
opt_uni  = optim.Adam(net_uni.parameters(), lr=LR_ADAPT)

x_uni_np = np.linspace(0, 1, N_INIT * N_ROUNDS + N_INIT, dtype=np.float32).reshape(-1, 1)
x_uni_t  = torch.from_numpy(x_uni_np)

loss_uni = []
for epoch in range(EPOCHS_PER * N_ROUNDS):
    loss_uni.append(train_step(net_uni, opt_uni, x_uni_t, x_ic_t, u_ic_t))

# ══════════════════════════════════════════════════════════════════════════════
#  Run 2: adaptive collocation
# ══════════════════════════════════════════════════════════════════════════════
net_ada  = make_net()
opt_ada  = optim.Adam(net_ada.parameters(), lr=LR_ADAPT)

x_ada_np = np.linspace(0, 1, N_INIT, dtype=np.float32).reshape(-1, 1)
x_ada_t  = torch.from_numpy(x_ada_np.copy())

x_cand_np = np.linspace(0, 1, N_CAND, dtype=np.float32).reshape(-1, 1)

loss_ada   = []
colloc_log = [x_ada_np.copy()]   # record collocation sets at each round

for rnd in range(N_ROUNDS):
    # Train for EPOCHS_PER epochs
    for epoch in range(EPOCHS_PER):
        loss_ada.append(train_step(net_ada, opt_ada, x_ada_t, x_ic_t, u_ic_t))

    # Evaluate residuals on candidate pool and select top-k
    x_cand_t = torch.from_numpy(x_cand_np).requires_grad_(True)
    with torch.enable_grad():
        res_cand = ode_residual(net_ada, x_cand_t).detach().numpy().flatten()
    top_k_idx = np.argsort(np.abs(res_cand))[-N_ADD:]
    x_new_np  = x_cand_np[top_k_idx]

    x_ada_np = np.vstack([x_ada_np, x_new_np])
    x_ada_t  = torch.from_numpy(x_ada_np.copy())
    colloc_log.append(x_ada_np.copy())
    print(f"Round {rnd+1}: added {N_ADD} points → total {len(x_ada_np)} collocation pts")

# ── Evaluate both networks ────────────────────────────────────────────────────
x_ev_t = torch.from_numpy(x_ev.reshape(-1, 1))
with torch.no_grad():
    pred_uni = net_uni(x_ev_t).numpy().flatten()
    pred_ada = net_ada(x_ev_t).numpy().flatten()

mse_uni = float(np.mean((pred_uni - u_ev)**2))
mse_ada = float(np.mean((pred_ada - u_ev)**2))

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: solution comparison
axes[0].plot(x_ev, u_ev,      'k-',  lw=2,   label='Exact $e^{-20x}$')
axes[0].plot(x_ev, pred_uni,  'r--', lw=1.8, label=f'Uniform  (MSE={mse_uni:.4f})')
axes[0].plot(x_ev, pred_ada,  'b-',  lw=1.8, label=f'Adaptive (MSE={mse_ada:.4f})')
axes[0].set_xlabel('x')
axes[0].set_ylabel('u(x)')
axes[0].set_title('Solution comparison', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Centre: training loss
axes[1].semilogy(loss_uni, 'r', lw=1.2, alpha=0.8, label='Uniform')
axes[1].semilogy(loss_ada, 'b', lw=1.2, alpha=0.8, label='Adaptive')
for i in range(1, N_ROUNDS + 1):
    axes[1].axvline(i * EPOCHS_PER, color='gray', lw=0.8, ls='--')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss (log scale)')
axes[1].set_title('Training loss\n(dashed lines = refinement steps)', fontsize=11, fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

# Right: collocation point distribution after final round
x_final = colloc_log[-1].flatten()
axes[2].hist(x_final, bins=20, color='royalblue', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('x')
axes[2].set_ylabel('Number of collocation points')
axes[2].set_title('Adaptive collocation distribution\n(concentrated near steep region)',
                  fontsize=11, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.suptitle('Adaptive vs uniform collocation  —  steep exponential $u = e^{-20x}$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal MSE — uniform   : {mse_uni:.6f}")
print(f"Final MSE — adaptive  : {mse_ada:.6f}")

### 2.4 Interpreting the results

The histogram in the right panel shows that after five refinement rounds the adaptive scheme concentrates collocation points near $x = 0$, where $e^{-20x}$ changes most rapidly. This is exactly where the ODE residual is largest during early training — the network has not yet captured the steep initial decay.

**Practical considerations**

| Issue | Remedy |
|---|---|
| Computational cost per epoch increases as points are added | Use a growing-batch schedule; freeze early-added points if memory is limited |
| Residual is high everywhere at initialisation — refinement is not informative | Delay the first refinement step until the loss has decreased by at least one order of magnitude |
| Points cluster too tightly around a single spike | Add a repulsion term or enforce a minimum inter-point distance |
| The candidate pool is too coarse | Increase $N_{\text{cand}}$ or use Latin hypercube sampling for the pool |

The residual-adaptive approach (RAR) described here is the simplest variant. More sophisticated methods include importance-weighted sampling (van der Meer et al., 2022) and gradient-enhanced residual sampling, which also considers the gradient of the residual with respect to the input coordinates.

## 3. Domain decomposition — XPINN sketch

### 3.1 Motivation

A single neural network is tasked with approximating the PDE solution over the entire space-time domain. For problems with:

- **Long time horizons** — the network must represent a solution that varies over many characteristic time scales. Standard PINNs exhibit *propagation failure*: the training signal from the initial condition decays before it reaches late times, so errors accumulate.
- **Complex geometries** — irregular boundaries increase the difficulty of placing collocation points and satisfying conditions everywhere.
- **Multiple scales** — solutions may contain both slow global trends and fast local features that cannot be captured by a single network of practical size.

Domain decomposition addresses these issues by partitioning the domain $\Omega$ into sub-domains $\{\Omega_i\}$ and training a separate sub-network $u_{\theta_i}$ on each, coupled through interface conditions that enforce continuity of the solution and its flux.

### 3.2 The XPINN framework

The *extended physics-informed neural network* (XPINN), proposed by Jagtap & Karniadakis (2020), decomposes the domain into $K$ non-overlapping (or slightly overlapping) sub-domains $\Omega_1, \ldots, \Omega_K$ with interfaces $\Gamma_{ij} = \partial\Omega_i \cap \partial\Omega_j$. Each sub-domain carries its own network $u_{\theta_i}$ and the total loss is

$$\mathcal{L}_{\text{XPINN}} = \sum_{i=1}^{K} \left[ \mathcal{L}^{(i)}_{\text{PDE}} + \mathcal{L}^{(i)}_{\text{BC}} \right] + \sum_{(i,j)} \mathcal{L}^{(ij)}_{\text{interface}}$$

where the interface loss enforces two conditions on $\Gamma_{ij}$:

**Continuity of the solution** (no jump in $u$):

$$\mathcal{L}^{(ij)}_{\text{avg}} = \frac{1}{N_{\Gamma}} \sum_{x \in \Gamma_{ij}} \left| u_{\theta_i}(x) - u_{\theta_j}(x) \right|^2$$

**Continuity of the flux** (no jump in the normal derivative):

$$\mathcal{L}^{(ij)}_{\text{flux}} = \frac{1}{N_{\Gamma}} \sum_{x \in \Gamma_{ij}} \left| \partial_n u_{\theta_i}(x) - \partial_n u_{\theta_j}(x) \right|^2$$

where $\partial_n$ denotes the outward normal derivative at the interface. For a 1D partition at $x = x^*$, the flux condition is simply $u'_{\theta_i}(x^*) = u'_{\theta_{i+1}}(x^*)$.

### 3.3 A 1D illustration

Consider the 1D heat equation on $[0,1] \times [0,4]$ (a long time domain). We split into two temporal sub-domains:

$$\Omega_1 = [0,1] \times [0,2], \qquad \Omega_2 = [0,1] \times [2,4]$$

with interface $\Gamma = \{t = 2\}$. Sub-network $u_{\theta_1}$ is trained on $\Omega_1$ with the original initial conditions. Sub-network $u_{\theta_2}$ is trained on $\Omega_2$ using the interface conditions at $t=2$ as its effective initial conditions. The diagram below illustrates the decomposition.

In [ ]:
# ── XPINN conceptual diagram — 1D temporal decomposition ──────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

# Sub-domain rectangles
rect1 = plt.Rectangle((0, 0), 1, 2, linewidth=2, edgecolor='royalblue',
                        facecolor='royalblue', alpha=0.18)
rect2 = plt.Rectangle((0, 2), 1, 2, linewidth=2, edgecolor='tomato',
                        facecolor='tomato', alpha=0.18)
ax.add_patch(rect1)
ax.add_patch(rect2)

# Borders
ax.plot([0, 1], [0, 0], 'k-',  lw=2.5)        # bottom (IC)
ax.plot([0, 1], [4, 4], 'k--', lw=1.5)        # top (final time)
ax.plot([0, 0], [0, 4], 'k-',  lw=2.5)        # left BC
ax.plot([1, 1], [0, 4], 'k-',  lw=2.5)        # right BC

# Interface
ax.plot([0, 1], [2, 2], 'g-', lw=3, label='Interface $\\Gamma$  ($t = 2$)')

# Labels
ax.text(0.5, 1.0,  '$\\Omega_1$\n$u_{\\theta_1}(x,t)$\n$t \\in [0,2]$',
        ha='center', va='center', fontsize=13, color='royalblue', fontweight='bold')
ax.text(0.5, 3.0,  '$\\Omega_2$\n$u_{\\theta_2}(x,t)$\n$t \\in [2,4]$',
        ha='center', va='center', fontsize=13, color='tomato', fontweight='bold')

# Interface conditions annotation
ax.annotate('', xy=(1.25, 2.3), xytext=(1.25, 1.7),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1.28, 2.0,
        'Interface conditions:\n'
        '$u_{\\theta_1}(x,2) = u_{\\theta_2}(x,2)$\n'
        "$\\partial_t u_{\\theta_1}(x,2) = \\partial_t u_{\\theta_2}(x,2)$",
        ha='left', va='center', fontsize=9.5, color='darkgreen')

# Initial condition annotation
ax.text(0.5, -0.22, 'Initial condition: $u(x, 0)$ prescribed',
        ha='center', va='center', fontsize=10, color='black')

ax.set_xlim(-0.05, 2.0)
ax.set_ylim(-0.4, 4.3)
ax.set_xlabel('$x$', fontsize=12)
ax.set_ylabel('$t$', fontsize=12)
ax.set_title('XPINN domain decomposition — temporal split at $t = 2$',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.set_aspect('equal')
ax.set_xticks([0, 0.5, 1])
ax.set_yticks([0, 1, 2, 3, 4])
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 3.4 Interface conditions in detail

For the heat equation $u_t = \kappa u_{xx}$ the physically correct interface conditions at a spatial interface $x = x^*$ are:

| Condition | Mathematical form | Physical meaning |
|---|---|---|
| Solution continuity | $u_{\theta_1}(x^*, t) = u_{\theta_2}(x^*, t)$ | No jump in temperature |
| Flux continuity | $\kappa \partial_x u_{\theta_1}(x^*, t) = \kappa \partial_x u_{\theta_2}(x^*, t)$ | Heat flux is conserved |

For a temporal interface at $t = t^*$ the flux condition involves the time derivative: $\partial_t u_{\theta_1}(x, t^*) = \partial_t u_{\theta_2}(x, t^*)$.

Both derivatives are available via autograd and enter the loss in the usual mean-squared-error form.

### 3.5 Advantages and limitations

**Advantages**
- Each sub-network is smaller and easier to train than a single global network.
- Sub-domains can in principle be trained in parallel, reducing wall-clock time.
- Propagation failure is mitigated because each sub-network covers only a short time interval.
- Different network architectures can be used in different regions (e.g., a deeper network near a shock).

**Limitations**
- Interface conditions introduce additional loss terms and hyperparameters.
- Choosing the decomposition requires physical insight — poor partitioning can create new pathologies.
- The interface weights must be tuned alongside the physics weight λ.
- Full implementations are significantly more complex than single-network PINNs.

A complete implementation of XPINN for the 1D heat equation would follow exactly the same autograd machinery as in [Session 7](Session7.ipynb) and [Session 8](Session8.ipynb), with the interface loss terms added to the optimisation objective. This is left as an exercise.

## 4. Recent advances — conceptual overview

The PINN landscape has evolved rapidly since the original Raissi et al. (2019) paper. The three directions below represent active research fronts as of 2024–2025. No code is provided — the goal is to give you the vocabulary to read the primary literature.

### 4.1 Conservative PINNs (cPINNs)

Standard PINNs enforce the *differential* form of the PDE at collocation points, which means conservation laws (mass, momentum, energy) are satisfied only in a pointwise, approximate sense. For problems where conservation is fundamental — such as the Euler equations of gas dynamics or the shallow-water equations in oceanography — this can lead to solutions that violate global balances even when the local residual is small.

Jagtap et al. (2020) introduced *conservative PINNs* (cPINNs) by working instead from the *integral* (weak) form of the conservation law. The domain is partitioned into control volumes, and on each control volume the residual is the net flux through the faces minus the source term — directly analogous to the finite-volume method familiar from numerical fluid dynamics. Because each control volume enforces a flux balance, global conservation is automatically satisfied up to the accuracy of the quadrature rule, regardless of the pointwise residual elsewhere.

**Key reference**: Jagtap, A. D. & Karniadakis, G. E. (2020). *Conservative physics-informed neural networks on discrete domains for conservation laws: applications to forward and inverse problems*. Computer Methods in Applied Mechanics and Engineering, 365, 113028.

### 4.2 Multi-fidelity PINNs

In many engineering and scientific settings, data are available at multiple levels of fidelity: a coarse computational fluid dynamics simulation might be cheap to run but inaccurate; a fine-grid simulation or laboratory measurement is accurate but expensive. *Multi-fidelity PINNs* (Meng & Karniadakis, 2020) exploit this structure by training a hierarchy of networks. A low-fidelity network $u^L_\theta$ is trained first on abundant cheap data; a high-fidelity correction network $u^H_\theta$ then learns the discrepancy $u^H - u^L$ from a small number of expensive observations. Physics constraints are applied to the full model $u^H$, not to $u^L$ in isolation.

This approach is closely related to multi-fidelity Gaussian-process emulators and composite neural operator methods. It is particularly relevant in high-energy physics and climate modelling, where the cost of high-fidelity simulations is the limiting factor.

**Key reference**: Meng, X. & Karniadakis, G. E. (2020). *A composite neural network that learns from multi-fidelity data: application to function approximation and inverse PDE problems*. Journal of Computational Physics, 401, 109020.

### 4.3 PIKANs — Physics-Informed Kolmogorov–Arnold Networks

Kolmogorov–Arnold Networks (KANs), introduced by Liu et al. (2024), replace the fixed activation functions of the standard multilayer perceptron (MLP) with learnable univariate spline functions on the edges of the network graph. The theoretical motivation comes from the Kolmogorov–Arnold representation theorem, which states that any continuous multivariate function can be written as a finite composition of continuous univariate functions. By learning the activation shapes rather than only the weights, KANs can represent certain structured functions with far fewer parameters than an equivalent MLP.

*Physics-Informed KANs* (PIKANs) embed KAN architectures into the PINN training loop: the network output and its derivatives (computed via autograd through the spline activations) enter the PDE residual in the usual way. Early empirical results (Wang et al., 2024; Shukla et al., 2024) suggest that PIKANs can achieve comparable or better accuracy than MLPs on certain smooth PDE problems, particularly when the solution has a known compositional structure. Whether this advantage persists for turbulent or high-dimensional problems remains an open question.

**Key references**:
- Liu, Z. et al. (2024). *KAN: Kolmogorov–Arnold Networks*. arXiv:2404.19756.
- Shukla, K. et al. (2024). *A comprehensive and FAIR comparison between MLP and KAN representations for differential equations and operator networks*. Computer Methods in Applied Mechanics and Engineering, 431, 117290.

## 5. Summary and look ahead

This session has covered three practical failure modes of standard PINNs and how to address them:

| Problem | Root cause | Remedy |
|---|---|---|
| High-frequency failures | Spectral bias of gradient descent | Fourier feature embeddings (RFF) |
| Wasted collocation budget | Uniform sampling ignores residual structure | Residual-adaptive refinement (RAR) |
| Propagation failure on long domains | Single global network cannot extrapolate in time | Domain decomposition (XPINN) |

Together with [Session 9](Session9.ipynb) (λ-tuning), these techniques form a toolkit for diagnosing and improving PINN performance on problems beyond the simple textbook setting.

The three research directions surveyed in Section 4 — cPINNs, multi-fidelity PINNs, and PIKANs — each address a distinct gap in the original PINN framework (conservation, data efficiency, and architectural expressivity, respectively). They are active areas of development and worth following in the literature.

### Looking ahead

[Session 11](Session11.ipynb) begins the applications thread of the course. We will apply PINNs to three physical systems:

- The **Schrödinger equation** (quantum mechanics) — a complex-valued PDE with a conserved probability current.
- The **Navier–Stokes equations** (fluid dynamics) — a coupled system in two spatial dimensions with incompressibility constraints.
- A **reaction–diffusion system** (pattern formation) — a two-component PDE that produces Turing patterns.

Each example will use the techniques developed in Sessions 7–10: loss balancing, Fourier features where appropriate, and residual monitoring to detect training pathologies early.

### Further reading

- Rahaman, N. et al. (2019). *On the spectral bias of neural networks*. ICML 2019.
- Tancik, M. et al. (2020). *Fourier features let networks learn high-frequency functions in low-dimensional domains*. NeurIPS 2020.
- Lu, L. et al. (2021). *DeepXDE: a deep learning library for solving differential equations*. SIAM Review, 63(1), 208–228. (RAR is described in Section 3.)
- Jagtap, A. D. & Karniadakis, G. E. (2020). *Extended physics-informed neural networks (XPINNs): a generalised space-time domain decomposition based deep learning framework for nonlinear partial differential equations*. Communications in Computational Physics, 28(5), 2002–2041.
- Jagtap, A. D. & Karniadakis, G. E. (2020). *Conservative physics-informed neural networks on discrete domains for conservation laws*. CMAME, 365, 113028.